In [ ]:
# %pip install transformers
# %pip install sentence-transformers
# %pip install scikit-learn
# %pip install textstat


In [ ]:
import pandas as pd
from datetime import datetime
import json
import re
from collections import Counter

In [ ]:
import nltk
import nltk.corpus
from nltk.corpus import stopwords
nltk.download('stopwords')

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

In [ ]:
# Load dataset
def load_dataset(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

In [ ]:
path = "/content/drive/MyDrive/Colab Notebooks/266/266_final_project/train.json"

data = load_dataset(path)

#### Dataset Overview
Number of examples, Unique sources, Markets, Date range, Missing values

In [ ]:
df = pd.DataFrame(data)
num_examples = len(df)


unique_sources = df["source"].unique()
unique_markets = df["market"].unique()

# Date range - Convert date column to datetime
df["date"] = pd.to_datetime(df["date"], errors="coerce")
date_min = df["date"].min()
date_max = df["date"].max()

# Missing values
missing_values = df.isna().sum()

In [ ]:
print("Number of examples:", num_examples)
print("\nUnique sources:", unique_sources)
print("\nUnique markets:", unique_markets)
print("\nDate range:", date_min, "to", date_max)
print("\nMissing values:\n", missing_values)

Number of examples: 3938

Unique sources: ['investrade' 'leaprate' 'totalfarmmarketing' 'vtmarkets']

Unique markets: ['equity market' 'gold' 'oil' 'treasury' 'currency' 'cattle' 'corn'
 'dairy' 'lean hog' 'soybean' 'wheat']

Date range: 2019-04-30 00:00:00 to 2023-01-04 00:00:00

Missing values:
 source         0
market         0
date           0
instruction    0
table_data     0
report         0
prompts        0
dtype: int64


In [ ]:
df.head()

,source,market,date,instruction,table_data,report,prompts
0,investrade,equity market,2020-08-31,Please act as an expert financial market analy...,Date Product Name ...,U.S. stocks finished the final trading day of ...,{'instruction': 'Please act as an expert finan...
1,investrade,equity market,2020-09-01,Please act as an expert financial market analy...,Date Product Name ...,"Lather Rinse Repeat – same recap, new day as t...",{'instruction': 'Please act as an expert finan...
2,investrade,equity market,2020-09-02,Please act as an expert financial market analy...,Date Product Name ...,Another day – another record high for stocks. ...,{'instruction': 'Please act as an expert finan...
3,investrade,equity market,2020-09-03,Please act as an expert financial market analy...,Date Product Name ...,It was bound to happen one of these days as U....,{'instruction': 'Please act as an expert finan...
4,investrade,equity market,2020-09-04,Please act as an expert financial market analy...,Date Product Name ...,A breath-taking sell-off on Thursday was follo...,{'instruction': 'Please act as an expert finan...


In [ ]:
print(df.shape)
df.info()

(3938, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3938 entries, 0 to 3937
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   source       3938 non-null   object        
 1   market       3938 non-null   object        
 2   date         3938 non-null   datetime64[ns]
 3   instruction  3938 non-null   object        
 4   table_data   3938 non-null   object        
 5   report       3938 non-null   object        
 6   prompts      3938 non-null   object        
dtypes: datetime64[ns](1), object(6)
memory usage: 215.5+ KB


Each daily market report is supported by a dataset of instruments traded or tracked on that date.
<br>The report narrative is written based on those instruments movements.
<br>Each market report correspond to a table of OHLCV for multiple products
<br> The dataset is not organized by pure asset class. It is organized by market narrative scope. Each daily report pulls in a basket of relevant instruments, not just instruments from one category.

In [ ]:
# print(df['table_data'][11]) # use print to see complete table data
# df['table_data'][11]

'      Date                     Product Name           Symbol     Open     High      Low    Close  Volume\n2020-08-19                          S&P 500            ^GSPC  3392.51  3399.54  3369.66  3374.85     NaN\n2020-08-20                          S&P 500            ^GSPC  3360.48  3390.80  3354.69  3385.51     NaN\n2020-08-21                          S&P 500            ^GSPC  3386.01  3399.96  3379.31  3397.16     NaN\n2020-08-24                          S&P 500            ^GSPC  3418.09  3432.09  3413.13  3431.28     NaN\n2020-08-25                          S&P 500            ^GSPC  3435.95  3444.21  3425.84  3443.62     NaN\n2020-08-26                          S&P 500            ^GSPC  3449.97  3481.07  3444.15  3478.73     NaN\n2020-08-27                          S&P 500            ^GSPC  3485.14  3501.38  3468.35  3484.55     NaN\n2020-08-28                          S&P 500            ^GSPC  3494.69  3509.23  3484.32  3508.01     NaN\n2020-08-31                          S&P 500  

#### 1. Linguistic analysis

Number of sentences, Number of words, Average sentence length, Average word, length, Vocabulary size, Type-token ratio, Most common words

In [ ]:
def extract_linguistic_features(text):
    """Extract basic linguistic features from a text."""

    # Sentence segmentation
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    sentences = [s for s in sentences if s]  # remove empty

    # Word tokenization
    words = re.findall(r'\b\w+\b', text.lower())

    num_sentences = len(sentences)
    num_words = len(words)

    # Vocabulary
    vocab = set(words)
    vocab_size = len(vocab)


    # Metrics
    avg_sentence_length = num_words / num_sentences if num_sentences else 0
    avg_word_length = sum(len(w) for w in words) / num_words if num_words else 0
    type_token_ratio = vocab_size / num_words if num_words else 0

    return {
        'num_sentences': num_sentences,
        'num_words': num_words,
        'avg_sentence_length': avg_sentence_length,
        'avg_word_length': avg_word_length,
        'vocab_size': vocab_size,
        'type_token_ratio': type_token_ratio
    }

In [ ]:
lin_features = df['report'].apply(extract_linguistic_features)

lin_features_df = pd.DataFrame(lin_features.tolist())

df_analysis = pd.concat([df.reset_index(drop=True), lin_features_df],axis=1)

New dataframe is df_analysis

In [ ]:
df_analysis.head()

,source,market,date,instruction,table_data,report,prompts,num_sentences,num_words,avg_sentence_length,avg_word_length,vocab_size,type_token_ratio
0,investrade,equity market,2020-08-31,Please act as an expert financial market analy...,Date Product Name ...,U.S. stocks finished the final trading day of ...,{'instruction': 'Please act as an expert finan...,7,158,22.571429,4.170886,99,0.626582
1,investrade,equity market,2020-09-01,Please act as an expert financial market analy...,Date Product Name ...,"Lather Rinse Repeat – same recap, new day as t...",{'instruction': 'Please act as an expert finan...,2,88,44.000000,4.477273,79,0.897727
2,investrade,equity market,2020-09-02,Please act as an expert financial market analy...,Date Product Name ...,Another day – another record high for stocks. ...,{'instruction': 'Please act as an expert finan...,3,93,31.000000,4.311828,70,0.752688
3,investrade,equity market,2020-09-03,Please act as an expert financial market analy...,Date Product Name ...,It was bound to happen one of these days as U....,{'instruction': 'Please act as an expert finan...,4,104,26.000000,4.557692,77,0.740385
4,investrade,equity market,2020-09-04,Please act as an expert financial market analy...,Date Product Name ...,A breath-taking sell-off on Thursday was follo...,{'instruction': 'Please act as an expert finan...,6,244,40.666667,4.536885,157,0.643443


In [ ]:
df_analysis.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3938 entries, 0 to 3937
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   source               3938 non-null   object        
 1   market               3938 non-null   object        
 2   date                 3938 non-null   datetime64[ns]
 3   instruction          3938 non-null   object        
 4   table_data           3938 non-null   object        
 5   report               3938 non-null   object        
 6   prompts              3938 non-null   object        
 7   num_sentences        3938 non-null   int64         
 8   num_words            3938 non-null   int64         
 9   avg_sentence_length  3938 non-null   float64       
 10  avg_word_length      3938 non-null   float64       
 11  vocab_size           3938 non-null   int64         
 12  type_token_ratio     3938 non-null   float64       
dtypes: datetime64[ns](1), float64(3),

Report-level word frequency across all reports:

In [ ]:
all_words = []

for report in df['report']:
    words = re.findall(r'\b\w+\b', report.lower())
    words = [w for w in words if w.isalpha()] # keep only alphabetic tokens drop numbers
    all_words.extend(words)

word_freq = Counter(all_words)
# word_freq.most_common(30)

In [ ]:
# remove stopwords before looking at the most common words
stop_words = set(stopwords.words('english'))

content_words = [word for word in all_words if word not in stop_words]

print(Counter(content_words).most_common(50))

[('week', 3078), ('futures', 2865), ('day', 2709), ('higher', 2580), ('market', 2579), ('cents', 2383), ('prices', 2111), ('lower', 1995), ('today', 1852), ('last', 1487), ('close', 1441), ('trading', 1388), ('trade', 1363), ('closing', 1349), ('index', 1345), ('cash', 1322), ('cattle', 1297), ('gained', 1263), ('session', 1258), ('year', 1171), ('price', 1134), ('average', 1131), ('dollar', 1123), ('lost', 1116), ('wheat', 1094), ('gains', 1088), ('may', 1047), ('high', 1022), ('dec', 1018), ('closed', 962), ('end', 931), ('level', 900), ('strong', 882), ('highs', 876), ('p', 871), ('oil', 869), ('hog', 863), ('since', 854), ('corn', 835), ('us', 834), ('still', 822), ('month', 818), ('support', 809), ('new', 789), ('contract', 787), ('back', 733), ('u', 703), ('stocks', 692), ('friday', 688), ('midday', 664)]


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:


# 2. Financial Entity Extraction (Simulating Evidence Extraction)
def extract_financial_entities(text):
    """Extract potential financial entities and their types."""
    # A simplified version using regex and known patterns
    patterns = {
        'TICKER': r'\b[A-Z]+\b',  # Needs refinement
        'PERCENTAGE': r'\d+\.?\d*%',
        'PRICE': r'\$\d+\.?\d*',
        'DATE': r'\d{1,2}/\d{1,2}/\d{2,4}|\d{4}-\d{2}-\d{2}',
        'BIG_NUMBER': r'\b\d{1,3}(?:,\d{3})*\b' # e.g., "1,000"
    }
    entities = {}
    for entity_type, pattern in patterns.items():
        matches = re.findall(pattern, text)
        if matches:
            entities[entity_type] = matches
    return entities

# 3. Sentiment Analysis (Using a domain-specific model like FinBERT)
def get_finbert_sentiment(text):
    """Get sentiment score using FinBERT."""
    tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
    model = AutoModel.from_pretrained("ProsusAI/finbert")
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    outputs = model(**inputs)
    # The output is a sequence of embeddings; you can pool them and use a classifier
    # For simplicity, we'll just return the pooled output.
    return outputs.pooler_output.detach().numpy().tolist()

# Main function to create the text analysis dataset
def prepare_text_dataset(json_data):
    text_analysis_data = []
    for entry in json_data:
        report = entry['report']
        text_analysis_data.append({
            'source': entry['source'],
            'market': entry['market'],
            'date': entry['date'],
            'report': report,
            'linguistic': extract_linguistic_features(report),
            'entities': extract_financial_entities(report),
            # 'sentiment': get_finbert_sentiment(report),  # Uncomment if you can run FinBERT
            'report_length': len(report),
            # Add a field for extracted "programs" if you implement it
        })
    return pd.DataFrame(text_analysis_data)

if __name__ == "__main__":
    # Load your processed dataset
    # Assuming you have 'train.json', 'val.json', 'test.json' in a directory
    data_dir = "data/processed_dataset/1month/"

    train_data = load_dataset(f"{data_dir}/train.json")
    val_data = load_dataset(f"{data_dir}/validate.json")
    test_data = load_dataset(f"{data_dir}/test.json")

    # Create analysis dataframes
    train_analysis_df = prepare_text_dataset(train_data)
    val_analysis_df = prepare_text_dataset(val_data)
    test_analysis_df = prepare_text_dataset(test_data)

    # Save dataframes for further analysis
    train_analysis_df.to_csv('train_text_analysis.csv', index=False)
    print("Text analysis complete.")